# Paper Reproduction Audit

This notebook reproduces the paper-facing Inventory and Cliff-Walking evidence from the repository, using runnable scripts kept next to the notebook:

```text
notebooks/reproduction/repro/inventory_train_repro.py
notebooks/reproduction/repro/cliff_train_repro.py
notebooks/reproduction/repro/figure_repro.py
notebooks/reproduction/repro/paper_targets.json
```

The comparison target is deliberately narrow and checkable:

1. Inventory and Cliff-Walking training curves.
2. Inventory decision-quality figure.
3. Cliff-Walking Appendix Table 3 numeric regret values.
4. Cliff-Walking Figure 2 decision-quality values extracted from the paper PDF.

The current notebook reproduces Inventory and Cliff Table 3 closely after using tuned reproduction parameters. The Cliff Figure 2 decision-quality panel is kept as a separate target because the Table 3 run does not reproduce it under the same hyperparameters.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / ".git").exists():
            return p
    raise RuntimeError(f"Could not find repo root from {start}")


REPO = find_repo_root(Path.cwd())
REPRO_DIR = REPO / "notebooks" / "reproduction" / "repro"
RESULTS_DIR = REPRO_DIR / "results"
SMOKE_DIR = RESULTS_DIR / "_smoke"
FIGURE_DIR = REPRO_DIR / "figures"
TARGETS_PATH = REPRO_DIR / "paper_targets.json"
PYTHON = sys.executable

print("repo:", REPO)
print("python:", PYTHON)
print("repro dir:", REPRO_DIR)

try:
    import qpth  # noqa: F401
except ModuleNotFoundError as exc:
    raise RuntimeError(
        "This notebook needs the project environment with qpth installed. "
        "On this machine, use the isolated conda environment `dfl_sa_cuda`."
    ) from exc

In [ ]:
def run_command(cmd, cwd=REPO, check=True):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd))
    proc = subprocess.run(
        cmd,
        cwd=cwd,
        text=True,
        encoding="utf-8",
        errors="replace",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(proc.stdout[-5000:])
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with code {proc.returncode}")
    return proc


def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

## 1. Smoke Tests

These cells are quick executable checks. They write under `results/_smoke/` and are not used as paper reproduction numbers.

### 1.1 Inventory Smoke

Runs the Inventory two-stage path on one seed.

In [ ]:
run_command([
    PYTHON,
    REPRO_DIR / "inventory_train_repro.py",
    "--results-root", SMOKE_DIR / "inventory",
    "--configs", "ts",
    "--seeds", "1",
    "--workers", "1",
    "--force",
])

### 1.2 Cliff Table 3 Smoke

Runs one Cliff Table 3 baseline method and one SA seed from scratch. This checks the Table 3 reproduction runner only.


In [ ]:
run_command([
    PYTHON,
    REPRO_DIR / "cliff_train_repro.py",
    "--results-root", SMOKE_DIR / "cliff",
    "--methods", "ts",
    "--n-seeds", "1",
    "--sa-specs", "6:0.5:500",
    "--sa-seeds", "0",
    "--workers", "1",
    "--force",
])


## 2. Full Reproduction Workflows

These cells run the full paper-comparison workflows. On a clean checkout they train all missing runs. If matching outputs already exist, the runners skip them; delete `repro/results/` or add `--force` to regenerate from scratch.

### 2.1 Inventory Full Reproduction

This runs TS, QP, LP, Sketch, and SA for `M = 10, 20, 30`. The QP recipe uses `qp_reg=50`, which is the tuned Inventory recipe recorded by this audit.

In [ ]:
run_command([
    PYTHON,
    REPRO_DIR / "inventory_train_repro.py",
    "--results-root", RESULTS_DIR / "inventory",
    "--configs", "ts", "lp", "qp_50", "sketch_r01", "sa10_r02", "sa20_r02", "sa30_r02",
    "--seeds", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10",
    "--workers", "3",
])

### 2.2 Cliff Table 3 Full Reproduction

This runner trains TS, QP, LP, Sketch, and the SA configurations from scratch for Appendix Table 3. It writes `results/cliff/cliff_combined_summary.json` and is intentionally separate from the Cliff decision-quality figure runner.


In [ ]:
run_command([
    PYTHON,
    REPRO_DIR / "cliff_train_repro.py",
    "--results-root", RESULTS_DIR / "cliff",
    "--methods", "ts", "qpth", "dfl", "sketch",
    "--n-seeds", "10",
    "--sa-specs", "6:0.5:500", "12:0.5:500", "24:2.0:500",
    "--sa-seeds", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9",
    "--workers", "1",
])

## 3. Results Display and Paper Comparison

This section displays the three paper-facing checks used in this audit:

1. training curves for Inventory and Cliff-Walking;
2. Inventory decision-quality figure and Cliff Figure 2 decision-quality target;
3. Cliff-Walking Appendix Table 3 numeric regret values.

The Cliff Figure 2 panel is not filled with Table 3 numbers. The Table 3 runner is a separate from-scratch reproduction target.


In [ ]:
if str(REPRO_DIR) not in sys.path:
    sys.path.insert(0, str(REPRO_DIR))

from figure_repro import (
    build_tables,
    plot_paper_vs_repro,
    plot_cliff_decision_figure_comparison,
    plot_cliff_table3_comparison,
    plot_training_curves,
    write_figures,
)

inventory_df, cliff_table_df, cliff_figure_df = build_tables(RESULTS_DIR, TARGETS_PATH)
comparison_df = pd.concat(
    [
        inventory_df.assign(task="Inventory decision figure"),
        cliff_figure_df.assign(task="Cliff decision figure"),
        cliff_table_df.assign(task="Cliff Table 3"),
    ],
    ignore_index=True,
)
cols = ["task", "method", "paper_mean", "paper_se", "repro_mean", "repro_se", "abs_error", "n", "status", "config"]
display(comparison_df[cols].round(4))

### Cliff Table 3 numeric reproduction

This is the Appendix Table 3 target by itself, separate from the Cliff decision-quality figure.

In [ ]:
display(cliff_table_df[cols[1:]].round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
plot_training_curves(RESULTS_DIR, axes[0], axes[1])
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
plot_paper_vs_repro(
    inventory_df,
    "Inventory MDP - Decision quality",
    "Test regret (% oracle reward)",
    axes[0],
)
plot_cliff_decision_figure_comparison(cliff_figure_df, axes[1])
fig.tight_layout()
plt.show()

### Cliff Table 3 plot

The figure below checks the exact numeric target from Appendix Table 3.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
plot_cliff_table3_comparison(cliff_table_df, ax)
fig.tight_layout()
plt.show()

In [ ]:
write_figures(RESULTS_DIR, TARGETS_PATH, FIGURE_DIR)
print("wrote figures to", FIGURE_DIR)

## 4. Reproduction Status, Parameters, and Missing Information


In [ ]:
param_rows = [
    {
        "target": "Inventory decision-quality figure",
        "paper_or_table2": "Table 2: qp_reg=1e-3; rho_QP=1e4; rho_LP=10; rho_Sketch=0.1; K=8; sketch_ratio=0.1; M={10,20,30}; tau=0.5",
        "reproduction_recipe": "qp_reg=50; rho_QP/rho_LP=1e4; rho_Sketch=10; K=8; sketch_ratio=0.1; SA rho=0.1; M={10,20,30}",
        "status": "Numerically reproducible, but only after changing key hyperparameters relative to Table 2.",
    },
    {
        "target": "Cliff Appendix Table 3",
        "paper_or_table2": "Table 2: lr=3e-3; batch=32; pretrain=60; epochs=50; seeds=10; rho_LP=0.1; rho_Sketch=1.0; rho_QP=0.1; qp_reg=0.1; K=8; sketch_ratio=0.2; tau=0.5",
        "reproduction_recipe": "From-scratch TS/QP/LP/Sketch plus SA; TS/QP/LP/Sketch use n_samples=200, train_frac=0.5, v4 data, loss_weight=500, qp_reg=0.1, Sketch K=4, sketch_ratio=0.1; SA uses M={6,12,24}, tau/loss_weight={6:0.5/500, 12:0.5/500, 24:2.0/500}; 10 seeds",
        "status": "Numerically reproducible from scratch; several reproduction hyperparameters differ from Table 2.",
    },
    {
        "target": "Cliff Figure 2 decision-quality panel",
        "paper_or_table2": "PDF target extracted from neurips_2026/figures/cw_decision_quality.pdf: TS=0.7379, QP=0.6740, LP=0.6897, Sketch=0.6754, SA12=8.0178, SA18=8.2250, SA24=0.7796",
        "reproduction_recipe": "No clean reproduction run found. Table 3 hyperparameters reproduce Table 3-scale regrets, not this Figure 2 scale/pattern; SA18 is also absent from the Table 3 reproduction run.",
        "status": "Not reproduced locally. I could not tune one shared Cliff recipe that simultaneously matches Figure 2 and Appendix Table 3.",
    },
    {
        "target": "Computation cost",
        "paper_or_table2": "Paper reports runtime/computation-cost comparisons.",
        "reproduction_recipe": "Not audited here because local hardware, CUDA/CPU usage, and solver runtime differ from the paper environment.",
        "status": "Not evaluated in this notebook.",
    },
]
param_df = pd.DataFrame(param_rows)
display(param_df)


### Conclusions

1. **Reproducibility status.** Inventory decision-quality and Cliff Appendix Table 3 can be reproduced closely with the runnable recipes in this notebook. The reproduced values and curves are close enough to treat these two targets as locally reproducible, but the required parameters are not identical to the paper/Table 2 settings.

2. **Cliff Figure 2 versus Table 3.** The Cliff-Walking Figure 2 decision-quality panel is not reproduced here. The Table 3 reproduction run gives Table 3-scale regret values and does not generate the Figure 2 values/pattern, including the low TS/QP/LP/Sketch bars and the SA M=18 result. In this local audit, I could not find a single shared Cliff hyperparameter setting that reproduces both Figure 2 and Appendix Table 3.

3. **Suspicious parameters needing confirmation.** The main parameters that need authorization/clarification are Inventory `qp_reg` and Sketch `aug_lag_rho`, plus the Cliff SA `tau` and `loss_weight` settings. These have large effects on the reproduced numbers and differ from, or are not fully specified by, the paper table.

4. **Parameters changed relative to the paper.** Inventory matches the decision-quality figure with `qp_reg=50` rather than Table 2 `qp_reg=1e-3`, and with Sketch `aug_lag_rho=10` rather than Table 2 `0.1`. Cliff Table 3 matches from scratch with Sketch `K=4,sketch_ratio=0.1` and SA recipes `M=6,tau=0.5,loss_weight=500`, `M=12,tau=0.5,loss_weight=500`, and `M=24,tau=2.0,loss_weight=500`; these should be checked against the intended final experiment settings.

5. **Missing or deferred pieces.** The exact command/raw numeric file for `cw_decision_quality.pdf` is still needed to reproduce Cliff Figure 2. Computation-cost reproduction is not included because local hardware and solver/runtime setup are not comparable to the paper environment.
